# Efecto del cambio de umbral

## Objetivo
Mantener fijas las probabilidades y observar cómo el umbral altera etiquetas, falsos positivos, falsos negativos, precisión, recall y F1. Datos sintéticos de alertas: clase 1 = caso que requiere revisión. No representa un sistema real.

| Umbral menor | Umbral mayor |
|---|---|
| Más alertas; recall no disminuye | Menos alertas; recall no aumenta |
| Falsos positivos no disminuyen | Falsos positivos no aumentan |
| Precisión puede subir o bajar | Precisión puede subir o bajar |

Separación 60 % entrenamiento, 20 % validación y 20 % test. El criterio didáctico fijado antes de ver resultados es minimizar 4×FN + FP: un FN cuesta cuatro unidades y un FP una. El costo debe definirse según el contexto; aquí es hipotético.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, ConfusionMatrixDisplay, PrecisionRecallDisplay
X,y=make_classification(n_samples=1200,n_features=8,n_informative=5,weights=[.75,.25],flip_y=.04,class_sep=.9,random_state=42)
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
X_train,X_val,y_train,y_val=train_test_split(X_dev,y_dev,test_size=.25,stratify=y_dev,random_state=43)
modelo=make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000)).fit(X_train,y_train)
p_val=modelo.predict_proba(X_val)[:,1]
def medidas(y,p,t):
    pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(umbral=t,TN=tn,FP=fp,FN=fn,TP=tp,precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred),F1=f1_score(y,pred),costo=4*fn+fp,alertas=int(pred.sum()))
tabla=pd.DataFrame([medidas(y_val,p_val,t) for t in np.round(np.arange(.05,.96,.05),2)])
display(tabla)
# Desempate previamente fijado: menor umbral, favoreciendo sensibilidad.
elegido=tabla.sort_values(['costo','umbral']).iloc[0]
t=float(elegido.umbral)
print('Umbral seleccionado en validación:',t)
assert (tabla.alertas.diff().dropna()<=0).all()
assert (tabla.recall.diff().dropna()<=0).all()

## Figuras y observaciones individuales
Las probabilidades no se recalculan al mover el umbral. Los gráficos siguientes usan exclusivamente validación.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
tabla.plot(x='umbral',y=['precision','recall','F1'],ax=axes[0]); axes[0].set_ylabel('Métrica')
tabla.plot(x='umbral',y='costo',ax=axes[1]); axes[1].axvline(t,color='black',linestyle='--'); axes[1].set_ylabel('4 × FN + FP'); plt.tight_layout(); plt.show()
PrecisionRecallDisplay.from_predictions(y_val,p_val); plt.title('Validación: curva precisión–recall'); plt.show()
casos=pd.DataFrame({'real':y_val[:12],'probabilidad':p_val[:12]})
for u in [.2,.5,.8]: casos[f'etiqueta_{u}']=(casos.probabilidad>=u).astype(int)
display(casos)

## Test final
Congelamos tanto el modelo como el umbral. No reajustamos el modelo después de elegir el umbral, porque eso podría cambiar sus probabilidades. Mostramos el umbral de referencia 0,50 y el elegido; no escogemos otra vez según test.

In [ ]:
p_test=modelo.predict_proba(X_test)[:,1]
display(pd.DataFrame([medidas(y_test,p_test,.5),medidas(y_test,p_test,t)],index=['Referencia 0,50','Elegido en validación']))
ConfusionMatrixDisplay.from_predictions(y_test,(p_test>=t).astype(int),display_labels=['Sin alerta','Revisión']); plt.title(f'Test; umbral fijado {t:.2f}'); plt.show()

## Práctica
1. Explique por qué la precisión no tiene que variar monótonamente.
2. Antes de una nueva ejecución, cambie el costo hipotético de FN de 4 a 2 y elija usando validación. No optimice mirando test.
3. Registre cuántas alertas cambiarían y qué consecuencia tendría cada error.

La calibración de probabilidades es una cuestión distinta del umbral. Fuente: [scikit-learn, ajuste del umbral](https://scikit-learn.org/stable/modules/classification_threshold.html).

## Galería de rendimiento y clasificación

Ejecute estas celdas después de las anteriores. Las imágenes y gráficos se generan con las predicciones reales de esta ejecución. El test se usa solo para diagnóstico del modelo ya seleccionado. No ajuste parámetros después de observar sus errores.

In [ ]:
# Dos reglas sobre las mismas probabilidades de test: referencia y umbral congelado.
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,u in zip(axes,[.5,t]):
    ConfusionMatrixDisplay.from_predictions(y_test,(p_test>=u).astype(int),labels=[0,1],display_labels=['Sin alerta','Revisión'],ax=ax,colorbar=False)
    ax.set_title(f'Test: umbral {u:.2f}')
plt.tight_layout(); plt.show()

fig,axes=plt.subplots(1,2,figsize=(12,4))
for clase,color in [(0,'#2468ac'),(1,'#b05220')]:
    axes[0].hist(p_val[y_val==clase],bins=np.linspace(0,1,16),alpha=.55,color=color,label=f'Clase real {clase}')
axes[0].axvline(t,color='black',linestyle='--',label=f'Umbral elegido {t:.2f}')
axes[0].set(xlabel='Probabilidad estimada',ylabel='Casos',title='Validación: distribución por clase real'); axes[0].legend()
tabla.plot(x='umbral',y=['FP','FN'],ax=axes[1]); axes[1].set(ylabel='Errores',title='Validación: errores al mover el umbral')
plt.tight_layout(); plt.show()

# Imagen de etiquetas predichas para los mismos casos de validación.
indices=np.argsort(p_val)  # ordenar solo para facilitar la lectura
umbrales=[.2,.5,.8]
imagen=np.vstack([(p_val[indices]>=u).astype(int) for u in umbrales])
fig,ax=plt.subplots(figsize=(12,2.5))
im=ax.imshow(imagen,aspect='auto',cmap='coolwarm',vmin=0,vmax=1,interpolation='nearest')
ax.set_yticks(range(3),[f'Umbral {u:.2f}' for u in umbrales])
ax.set(xlabel='Casos de validación ordenados por probabilidad',title='Mismas probabilidades, distintas etiquetas')
fig.colorbar(im,ax=ax,ticks=[0,1],label='Clase predicha'); plt.tight_layout(); plt.show()

### Interprete las figuras
Identifique las clases más difíciles, describa dos patrones visibles y distinga los gráficos de validación de los de test. Si propone un cambio del modelo, deberá evaluarlo con una nueva separación independiente; no convierta el test observado en un conjunto de selección.